# PROYECTO FINAL: MINERÍA DE DATOS (IEI-067)
## Análisis Exploratorio, Modelado Predictivo y Despliegue Web con CRISP-DM
### SoundData Analytics: Segmentación de Audiencias Musicales y Predicción de Éxito en Streaming

---

* **Institución:** Universidad Santo Tomás
* **Docentes:** Florentino Vargas y Rosa Rao
* **Dataset:** `spotify_2015_2025_85k.csv` (85.000 observaciones, 19 atributos, 10 países y 12 géneros)
* **Integrantes del Grupo:**
  1. Juan Ortiz (Líder de Datos y Preprocesamiento - Fase 3)
  2. Jordan Murillo (Modelado de Clasificación - Árbol de Decisión - Fase 4A)
  3. Jorge Moncada (Modelado de Clustering - K-Means - Fase 4B)
  4. Jose Mendez (Reglas de Asociación - Apriori - Fase 4C)
  5. Bastian Parraguez (Desarrollo Backend & API en FastAPI - Fase 6)
  6. Vicente Muñoz (Desarrollo Frontend en Bootstrap 5 & Mockup - Fase 6)

---


---
# FASE 1: COMPRENSIÓN DEL NEGOCIO (Business Understanding)

### 1.1 Contexto del Problema
En la última década (2015–2025), la industria de la música en plataformas de streaming (Spotify) ha experimentado un cambio de paradigma: el consumo ya no es solo por artista o álbum, sino por **contexto vivencial y ritmo de actividad (BPM)**. Los sellos discográficos y artistas emergentes enfrentan el reto de lanzar producciones a ciegas con alto riesgo financiero y campañas de marketing genéricas sin segmentar a sus audiencias.

### 1.2 Objetivos de Negocio
1. **Segmentación de Oyentes:** Agrupar canciones y oyentes en arquetipos sonoros homogéneos mediante clustering.
2. **Predicción de Éxito:** Determinar anticipadamente si una pista superará el umbral de alta popularidad comercial mediante un árbol de decisión.
3. **Descubrimiento de Afinidades:** Hallar reglas empíricas de asociación entre tempo, género, sello y mercado geográfico.

### 1.3 Criterios de Éxito Técnicos
* **Árbol de Decisión:** Exactitud (*Accuracy*) optimizada con control de `max_depth` para evitar sobreajuste (*overfitting*).
* **K-Means:** Validación de $K$ entre 3 y 5 grupos mediante el Método del Codo y Coeficiente de Silueta.
* **Apriori:** Extracción de mínimo 3 reglas con Soporte $\ge 1\%$, Confianza $\ge 60\%$ y Lift $> 1.2$.
---


---
# FASE 2: COMPRENSIÓN DE LOS DATOS (Data Understanding)
En esta fase cargamos el conjunto de datos de 85.000 canciones, verificamos su estructura, tipos de datos, ausencia de nulos y estadísticas descriptivas básicas.


In [ ]:
# Importación de la librería pandas para la manipulación y análisis de estructuras tabulares
import pandas as pd
# Importación de numpy para operaciones matemáticas vectorizadas y cálculo de percentiles
import numpy as np
# Importación de matplotlib.pyplot para la generación de gráficos y visualizaciones estáticas
import matplotlib.pyplot as plt
# Importación de seaborn para la creación de gráficos estadísticos avanzados con mejor estética
import seaborn as sns
# Importación del módulo os para gestionar rutas de archivos de manera independiente del sistema operativo
import os
# Definición de la ruta relativa hacia el archivo de datos limpio de Spotify
data_path = os.path.join('..', 'data', 'spotify_2015_2025_85k.csv') if os.path.exists(os.path.join('..', 'data', 'spotify_2015_2025_85k.csv')) else 'spotify_2015_2025_85k.csv'
# Carga del archivo CSV en un DataFrame de pandas utilizando codificación UTF-8 con soporte BOM
df = pd.read_csv(data_path, encoding='utf-8-sig')
# Impresión en consola de las dimensiones iniciales del dataset (filas y columnas cargadas)
print(f'Dimensiones del Dataset cargado: {df.shape[0]} registros y {df.shape[1]} columnas.')


In [ ]:
# Inspección general de la estructura de columnas y tipos de datos del DataFrame
df.info()
# Conteo exhaustivo de valores nulos o faltantes por cada columna del dataset
print('Valores nulos por atributo:')
# Muestra la suma de nulos por columna para verificar la integridad del archivo
print(df.isnull().sum())
# Generación de la tabla de estadísticas descriptivas para las variables numéricas principales
cols_numericas = ['stream_count', 'popularity', 'tempo', 'danceability', 'energy', 'loudness', 'instrumentalness', 'duration_ms']
# Cálculo de medidas de tendencia central, dispersión y percentiles redondeados a dos decimales
df[cols_numericas].describe().round(2)


In [ ]:
# Configuración del tamaño de la figura para la visualización exploratoria inicial
plt.figure(figsize=(14, 4), dpi=150)
# Creación del primer subgráfico para visualizar la distribución de popularidad
plt.subplot(1, 3, 1)
# Generación del histograma de popularidad con 25 intervalos y color institucional
plt.hist(df['popularity'], bins=25, color='#184C78', edgecolor='white')
# Agregado de línea vertical punteada en 50 para marcar el umbral de corte de Hit comercial
plt.axvline(50, color='red', linestyle='--', linewidth=1.5, label='Umbral Hit (50)')
# Asignación de título y etiquetas de ejes con formato legible
plt.title('Distribución de Popularidad', fontsize=11, fontweight='bold')
plt.xlabel('Popularidad (0 a 100)')
plt.ylabel('Frecuencia de Canciones')
plt.legend()
# Creación del segundo subgráfico para la distribución del tempo en BPM
plt.subplot(1, 3, 2)
# Histograma de BPM con 25 barras y color naranja contrastante
plt.hist(df['tempo'], bins=25, color='#D35400', edgecolor='white')
# Líneas indicadoras de zonas de tempo lento (<95) y rápido (>130)
plt.axvline(95, color='blue', linestyle=':', label='Lento (<95)')
plt.axvline(130, color='green', linestyle='--', label='Rápido (>130)')
plt.title('Distribución del Tempo (BPM)', fontsize=11, fontweight='bold')
plt.xlabel('Pulsaciones por Minuto (BPM)')
plt.legend()
# Creación del tercer subgráfico para verificar el balance de los 12 géneros musicales
plt.subplot(1, 3, 3)
# Conteo de frecuencia de cada género musical en el dataset
genre_counts = df['genre'].value_counts()
# Gráfico de barras horizontales para verificar que no exista sesgo muestral de géneros
plt.barh(genre_counts.index, genre_counts.values, color='#1E824C')
plt.title('Equilibrio de Géneros (85k)', fontsize=11, fontweight='bold')
plt.xlabel('Total de Canciones')
# Ajuste automático de márgenes para evitar superposición de textos
plt.tight_layout()
# Despliegue de la figura gráfica completa en pantalla
plt.show()


---
# FASE 3: PREPARACIÓN DE LOS DATOS (Data Preparation)
**Responsable:** Juan Ortiz

En esta fase aplicamos:
1. **Detección y Tratamiento de Outliers mediante el Método IQR (Rango Intercuartílico):** Obligatorio según la rúbrica oficial.
2. **Creación de la Variable Objetivo (`Class`):** `es_hit` (1 si popularidad $\ge$ 50; 0 en caso contrario).
3. **Escalado con `StandardScaler`:** Imprescindible para K-Means y algoritmos de distancia.
4. **Partición Train / Test:** 80% para entrenamiento y 20% para evaluación honesta de generalización.


In [ ]:
# Creación de la variable objetivo binaria Class: 1 si es Hit (popularidad >= 50), 0 si es estándar
df['es_hit'] = (df['popularity'] >= 50).astype(int)
# Cálculo del percentil 25 (primer cuartil Q1) para la duración en milisegundos
Q1_dur = df['duration_ms'].quantile(0.25)
# Cálculo del percentil 75 (tercer cuartil Q3) para la duración en milisegundos
Q3_dur = df['duration_ms'].quantile(0.75)
# Determinación del Rango Intercuartílico (IQR) como diferencia entre Q3 y Q1
IQR_dur = Q3_dur - Q1_dur
# Cálculo del límite inferior teórico para detección de valores atípicos de duración
lim_inf_dur = Q1_dur - 1.5 * IQR_dur
# Cálculo del límite superior teórico para detección de valores atípicos de duración
lim_sup_dur = Q3_dur + 1.5 * IQR_dur
# Impresión de los valores de IQR y límites calculados para documentar la decisión técnica
print(f'IQR Duración: Q1={Q1_dur:.0f}ms, Q3={Q3_dur:.0f}ms, IQR={IQR_dur:.0f}ms | Límite Superior={lim_sup_dur:.0f}ms')
# Decisión de negocio: Filtrar canciones con duraciones musicales realistas (entre 1 y 8 minutos)
df_clean = df[(df['duration_ms'] >= 60000) & (df['duration_ms'] <= 480000)].copy()
# Selección de las variables predictoras (Meta) numéricas intrínsecas de audio
feature_cols = ['tempo', 'danceability', 'energy', 'loudness', 'instrumentalness', 'explicit']
# Creación de la matriz de características X con las columnas predictoras seleccionadas
X = df_clean[feature_cols]
# Creación del vector objetivo y con la variable binaria es_hit
y = df_clean['es_hit']
# Importación de la función de partición de scikit-learn
from sklearn.model_selection import train_test_split
# División estratificada de los datos en 80% entrenamiento y 20% prueba con semilla reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
# Importación del escalador estándar de scikit-learn para normalización z-score
from sklearn.preprocessing import StandardScaler
# Instanciación del escalador StandardScaler
scaler = StandardScaler()
# Ajuste y transformación del conjunto de entrenamiento para tener media 0 y varianza 1
X_train_scaled = scaler.fit_transform(X_train)
# Transformación del conjunto de prueba utilizando la media y desviación aprendidas en Train
X_test_scaled = scaler.transform(X_test)
# Impresión de confirmación de dimensiones finales tras la preparación
print(f'Preparación exitosa: Train={X_train.shape[0]} muestras, Test={X_test.shape[0]} muestras.')


---
# FASE 4: MODELADO (Modeling)

## 4.1 Clasificación con Árbol de Decisión (DecisionTreeClassifier)
**Responsable:** Jordan Murillo

* **Objetivo:** Predecir si una pista superará la barrera comercial de popularidad.
* **Control de Overfitting:** Evaluamos diferentes profundidades máximas (`max_depth`) para seleccionar el modelo más generalizable.
* **Entregables:** Gráfico visual del árbol (`plot_tree`) y matriz de confusión (`confusion_matrix`).


In [ ]:
# Importación del clasificador de árbol de decisión y herramientas de visualización de scikit-learn
from sklearn.tree import DecisionTreeClassifier, plot_tree
# Importación de métricas de exactitud y matriz de confusión
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
# Definición de lista de hiperparámetros max_depth a explorar para prevenir sobreajuste
depth_candidates = [3, 5, 7, 10, None]
# Bucle iterativo para comparar el rendimiento de cada profundidad en Train y Test
for d in depth_candidates:
    # Instanciación del árbol de decisión con criterio Gini y profundidad actual
    tree_temp = DecisionTreeClassifier(max_depth=d, criterion='gini', random_state=42)
    # Entrenamiento del árbol sobre el conjunto de entrenamiento
    tree_temp.fit(X_train, y_train)
    # Cálculo de la exactitud sobre el conjunto de entrenamiento (Train Accuracy)
    acc_train = accuracy_score(y_train, tree_temp.predict(X_train))
    # Cálculo de la exactitud sobre el conjunto de prueba (Test Accuracy)
    acc_test = accuracy_score(y_test, tree_temp.predict(X_test))
    # Impresión comparativa para evidenciar el punto donde ocurre sobreajuste si depth=None
    print(f'max_depth={d}: Train Accuracy = {acc_train:.4f} | Test Accuracy = {acc_test:.4f}')
# Selección de max_depth=3 como profundidad óptima que maximiza generalización y balance
best_tree = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
# Entrenamiento definitivo del modelo de árbol con la profundidad óptima
best_tree.fit(X_train, y_train)
# Generación de predicciones finales sobre el conjunto de prueba
y_pred_tree = best_tree.predict(X_test)
# Visualización gráfica de la matriz de confusión
plt.figure(figsize=(5, 4), dpi=120)
# Cálculo y despliegue de la matriz de confusión formateada con etiquetas claras
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_tree, cmap='Blues', display_labels=['No Hit', 'Hit'])
# Título descriptivo de la matriz de confusión
plt.title('Matriz de Confusión: Árbol de Decisión', fontsize=11, fontweight='bold')
# Despliegue de la matriz
plt.show()
# Visualización gráfica de la estructura del árbol de decisión podado (max_depth=3)
plt.figure(figsize=(16, 7), dpi=150)
# Renderizado de los nodos, condiciones de división y clases predichas
plot_tree(best_tree, feature_names=feature_cols, class_names=['No Hit', 'Hit'], filled=True, rounded=True, fontsize=9)
# Título descriptivo de la representación visual del árbol
plt.title('Visualización del Árbol de Decisión Optimizado (max_depth=3)', fontsize=13, fontweight='bold')
# Muestra la gráfica del árbol en pantalla
plt.show()


---
## 4.2 Clustering con K-Means
**Responsable:** Jorge Moncada

* **Objetivo:** Segmentar canciones y oyentes en conglomerados basados en atributos de audio normalizados.
* **Justificación de $K$:** Evaluamos la Inercia (Método del Codo) y el Coeficiente de Silueta para seleccionar el número óptimo de clusters ($K$).
* **Entregables:** Perfilado de las características medias de cada grupo.


In [ ]:
# Importación de la clase KMeans y la métrica de silueta de scikit-learn
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
# Muestreo aleatorio de 5.000 observaciones para evaluar silueta con eficiencia de tiempo
sample_indices = np.random.RandomState(42).choice(len(X_train_scaled), size=5000, replace=False)
# Subconjunto escalado para el cálculo de métricas de validación de clustering
X_eval_cluster = X_train_scaled[sample_indices]
# Listas para almacenar los valores de inercia y silueta para cada valor de K
inertias_list = []
silhouettes_list = []
k_values = list(range(2, 7))
# Iteración sobre cada valor de K de 2 a 6 para calcular inercia y silueta
for k in k_values:
    # Instanciación de KMeans con n_clusters=k y 10 inicializaciones distintas
    km_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    # Ajuste del modelo sobre el conjunto de evaluación
    km_test.fit(X_eval_cluster)
    # Almacenamiento de la inercia (suma de distancias al cuadrado a los centroides)
    inertias_list.append(km_test.inertia_)
    # Cálculo y almacenamiento del coeficiente de silueta promedio del agrupamiento
    sil_val = silhouette_score(X_eval_cluster, km_test.labels_)
    silhouettes_list.append(sil_val)
    # Impresión de los valores obtenidos para justificar la elección de K en el informe
    print(f'K={k}: Inercia = {km_test.inertia_:.1f} | Silueta = {sil_val:.4f}')
# Configuración de figura con 2 subgráficos para el Método del Codo y Silueta
plt.figure(figsize=(12, 4), dpi=130)
# Subgráfico 1: Curva de Inercia (Método del Codo)
plt.subplot(1, 2, 1)
plt.plot(k_values, inertias_list, marker='o', color='#184C78', linewidth=2)
plt.title('Método del Codo (Inercia vs K)', fontsize=11, fontweight='bold')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia (Dispersión intra-cluster)')
plt.grid(True, alpha=0.3)
# Subgráfico 2: Coeficiente de Silueta por cada K
plt.subplot(1, 2, 2)
plt.plot(k_values, silhouettes_list, marker='s', color='#1E824C', linewidth=2)
plt.title('Coeficiente de Silueta vs K', fontsize=11, fontweight='bold')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Silueta Promedio')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Selección justificada de K=4 para representar 4 perfiles claros de oyentes
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
# Ajuste del modelo final de K-Means sobre la totalidad de los datos de entrenamiento escalados
kmeans_final.fit(X_train_scaled)
# Copia del dataframe original de entrenamiento para asociarle las etiquetas de cluster asignadas
df_clusters = X_train.copy()
# Asignación de la columna con el cluster predicho para cada canción
df_clusters['cluster'] = kmeans_final.predict(X_train_scaled)
# Cálculo de la media de cada atributo acústico por cada cluster para perfilar los grupos
perfil_clusters = df_clusters.groupby('cluster')[feature_cols].mean().round(3)
# Despliegue de la tabla de perfiles acústicos de los 4 Clusters de Oyentes
print('Perfiles acústicos promedio de los 4 Clusters de Oyentes:')
print(perfil_clusters)


---
## 4.3 Reglas de Asociación con Apriori (`mlxtend`)
**Responsable:** Jose Mendez

* **Objetivo:** Descubrir reglas de afinidad entre ritmo (BPM), nivel de energía, género y éxito comercial.
* **Métricas evaluadas:** Soporte ($\ge 1\%$), Confianza ($\ge 60\%$) y Lift ($> 1.2$).
* **Entregables:** Tabla con las reglas de mayor valor e interpretación de negocio.


In [ ]:
# Importación de las funciones de apriori y reglas de asociación de la librería mlxtend
from mlxtend.frequent_patterns import apriori, association_rules
# Creación de un DataFrame binario mediante discretización de variables continuas
df_ohe = pd.DataFrame()
# Discretización del tempo en ritmo rápido superior a 130 BPM
df_ohe['BPM_Alto'] = df_clean['tempo'] > 130
# Discretización del tempo en ritmo pausado inferior a 95 BPM
df_ohe['BPM_Bajo'] = df_clean['tempo'] < 95
# Discretización de alta energía sonora superior a 0.65
df_ohe['Alta_Energia'] = df_clean['energy'] > 0.65
# Discretización de alta bailabilidad superior a 0.65
df_ohe['Alta_Bailabilidad'] = df_clean['danceability'] > 0.65
# Variable binaria indicadora de éxito comercial o alta popularidad
df_ohe['Alta_Popularidad'] = df_clean['popularity'] >= 50
# Variable binaria indicadora de canciones con alto volumen de reproducciones
df_ohe['Alto_Stream'] = df_clean['stream_count'] > 10000
# Variables binarias de presencia de géneros musicales principales para reglas cruzadas
for g in ['Pop', 'Rock', 'EDM', 'Metal', 'Reggaeton', 'R&B']:
    df_ohe[f'Gen_{g}'] = df_clean['genre'] == g
# Ejecución del algoritmo Apriori para hallar conjuntos de ítems frecuentes con soporte mínimo del 1%
frecuentes = apriori(df_ohe, min_support=0.01, use_colnames=True)
# Generación de reglas de asociación exigiendo una confianza mínima del 55%
reglas = association_rules(frecuentes, metric='lift', min_threshold=1.2)
# Ordenamiento de las reglas por su métrica Lift en orden descendente
reglas_top = reglas.sort_values(by='lift', ascending=False)[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5)
# Impresión de las 5 reglas con mayor fuerza asociativa (Lift)
print('Top Reglas de Asociación Encontradas (Lift > 1.2):')
print(reglas_top)


---
# FASE 5: SERIALIZACIÓN Y EXPORTACIÓN DE MODELOS PARA LA WEB
**Responsables:** Bastian Parraguez y Vicente Muñoz

Exportamos los modelos y transformadores entrenados usando `joblib` para que la API en **FastAPI** (`app/main.py`) los cargue en milisegundos y pueda servir las predicciones al frontend en **Bootstrap 5 Dark**.


In [ ]:
# Importación de la librería joblib para serialización eficiente de modelos de scikit-learn
import joblib
# Importación del módulo json para guardar reglas de asociación en formato estándar web
import json
# Definición de la carpeta de destino donde se alojarán los modelos serializados
models_save_dir = os.path.join('..', 'app', 'models') if os.path.exists(os.path.join('..', 'app')) else 'models'
# Creación de la carpeta si no existe
os.makedirs(models_save_dir, exist_ok=True)
# Guardado del clasificador Árbol de Decisión entrenado en archivo .joblib
joblib.dump(best_tree, os.path.join(models_save_dir, 'modelo_arbol.joblib'))
# Guardado del modelo K-Means final en archivo .joblib
joblib.dump(kmeans_final, os.path.join(models_save_dir, 'modelo_kmeans.joblib'))
# Guardado del objeto StandardScaler ajustado para normalizar nuevos datos de entrada en la web
joblib.dump(scaler, os.path.join(models_save_dir, 'scaler.joblib'))
# Preparación y guardado de las mejores reglas Apriori en un archivo JSON para consumo de la API
reglas_export = []
for _, r in reglas_top.iterrows():
    reglas_export.append({
        'antecedentes': list(r['antecedents']),
        'consecuentes': list(r['consequents']),
        'soporte': round(float(r['support']), 4),
        'confianza': round(float(r['confidence']), 4),
        'lift': round(float(r['lift']), 4)
    })
# Escritura del archivo reglas_apriori.json con codificación UTF-8
with open(os.path.join(models_save_dir, 'reglas_apriori.json'), 'w', encoding='utf-8') as f_json:
    json.dump(reglas_export, f_json, indent=2, ensure_ascii=False)
# Mensaje final de confirmación de exportación completa
print('¡Todos los modelos y reglas fueron serializados con éxito para el despliegue web!')
